# Week 25, Day 5: Project: polish

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w025d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('money.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading and showing money and categories."""\nimport re\n\nAMOUNT = re.compile(r"£?\\s*(\\d{1,3}(?:,\\d{3})+|\\d+)(?:\\.(\\d{1,2}))?")\n\n\ndef parse_amount(text):\n    """Text such as \'£1,250.50\' as whole pence. Raises ValueError with a readable message."""\n    text = text.strip()\n    if not text.lstrip("£").strip():\n        raise ValueError("missing amount")\n    m = AMOUNT.fullmatch(text)\n    if m is None:\n        raise ValueError(f"not an amount: {text!r}")\n    total = int(m.group(1).replace(",", "")) * 100 + int((m.group(2) or "").ljust(2, "0"))\n    if total == 0:\n        raise ValueError("amount must be more than £0")\n    return total\n\n\ndef format_pence(pence):\n    """1250 as \'£12.50\', -350 as \'-£3.50\'."""\n    sign = "-" if pence < 0 else ""\n    pounds, rest = divmod(abs(pence), 100)\n    return f"{sign}£{pounds:,}.{rest:02d}"\n\n\ndef clean_category(text):\n    """\' Eating   Out \' as \'eating out\'. Raises ValueError if it isn\'t words of letters."""\n    shown = text.strip()\n    if not shown:\n        raise ValueError("missing category")\n    cleaned = " ".join(shown.lower().split())\n    if len(cleaned) > 20 or not re.fullmatch(r"[a-z]+( [a-z]+)*", cleaned):\n        raise ValueError(f"bad category: {shown!r}")\n    return cleaned\n')
with open('dates.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading dates and months that people type, and stepping through months."""\nimport re\nfrom datetime import datetime, timedelta\n\nFORMATS = ["%Y-%m-%d", "%d/%m/%Y"]\n\n\ndef parse_date(text, today):\n    """\'2026-11-02\', \'02/11/2026\', \'today\', \'yesterday\' or \'\' as ISO text. No future dates."""\n    shown = text.strip()\n    word = shown.lower()\n    if word in ("", "today"):\n        when = today\n    elif word == "yesterday":\n        when = today - timedelta(days=1)\n    else:\n        when = None\n        for fmt in FORMATS:\n            try:\n                when = datetime.strptime(word, fmt).date()\n                break\n            except ValueError:\n                pass\n        if when is None:\n            raise ValueError(f"not a date: {shown!r}")\n    if when > today:\n        raise ValueError(f"date is in the future: {when.isoformat()}")\n    return when.isoformat()\n\n\ndef parse_month(text):\n    """\'2026-11\' or \'11/2026\' as \'2026-11\'. Raises ValueError otherwise."""\n    shown = text.strip()\n    m = re.fullmatch(r"(\\d{4})-(\\d{2})", shown)\n    if m:\n        year, month = int(m.group(1)), int(m.group(2))\n    else:\n        m = re.fullmatch(r"(\\d{1,2})/(\\d{4})", shown)\n        if not m:\n            raise ValueError(f"not a month: {shown!r}")\n        month, year = int(m.group(1)), int(m.group(2))\n    if not 1 <= month <= 12:\n        raise ValueError(f"not a month: {shown!r}")\n    return f"{year}-{month:02d}"\n\n\ndef next_month(month):\n    """\'2026-12\' gives \'2027-01\'."""\n    year, m = int(month[:4]), int(month[5:])\n    if m == 12:\n        return f"{year + 1}-01"\n    return f"{year}-{m + 1:02d}"\n')
with open('core.py', 'w', encoding='utf-8') as f:
    f.write('"""The expense tracker\'s data and the operations on it."""\nfrom money import parse_amount, clean_category\nfrom dates import parse_date, next_month\n\n\ndef new_data():\n    return {"expenses": [], "budgets": {}, "next_id": 1}\n\n\ndef add_expense(data, amount_text, category_text, note, date_text, today):\n    """Check every field, then add the expense, or raise one ValueError listing every problem."""\n    problems = []\n    amount = category = when = None\n    try:\n        amount = parse_amount(amount_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        category = clean_category(category_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        when = parse_date(date_text, today)\n    except ValueError as e:\n        problems.append(str(e))\n    note = note.strip()\n    if len(note) > 60:\n        problems.append("note is longer than 60 characters")\n    if problems:\n        raise ValueError("; ".join(problems))\n    expense = {"id": data["next_id"], "date": when, "amount": amount, "category": category, "note": note}\n    data["expenses"].append(expense)\n    data["next_id"] += 1\n    return expense\n\n\ndef remove_expense(data, expense_id):\n    """Remove and return the expense with this id. KeyError if there isn\'t one."""\n    for i, e in enumerate(data["expenses"]):\n        if e["id"] == expense_id:\n            return data["expenses"].pop(i)\n    raise KeyError(f"no expense #{expense_id}")\n\n\ndef filter_expenses(data, month=None, category=None):\n    """A new list of the matching expenses, newest first (by date, then id)."""\n    chosen = [e for e in data["expenses"]\n              if (month is None or e["date"][:7] == month)\n              and (category is None or e["category"] == category)]\n    return sorted(chosen, key=lambda e: (e["date"], e["id"]), reverse=True)\n\n\ndef total(expenses):\n    return sum(e["amount"] for e in expenses)\n\n\ndef by_category(expenses):\n    """(category, pence) pairs, largest first, alphabetical for ties."""\n    totals = {}\n    for e in expenses:\n        totals[e["category"]] = totals.get(e["category"], 0) + e["amount"]\n    return sorted(totals.items(), key=lambda pair: (-pair[1], pair[0]))\n\n\ndef month_summary(data, month):\n    """Count, total, a row per category with spending or a budget, and the categories over budget."""\n    expenses = filter_expenses(data, month=month)\n    spent = dict(by_category(expenses))\n    rows = []\n    for category in set(spent) | set(data["budgets"]):\n        budget = data["budgets"].get(category)\n        used = spent.get(category, 0)\n        rows.append({"category": category, "spent": used, "budget": budget,\n                     "left": None if budget is None else budget - used})\n    rows.sort(key=lambda r: (-r["spent"], r["category"]))\n    return {"month": month, "count": len(expenses), "total": total(expenses), "rows": rows,\n            "over": [r["category"] for r in rows if r["left"] is not None and r["left"] < 0]}\n\n\ndef monthly_totals(data):\n    """(month, pence) for every month from the first with spending to the last, gaps as 0."""\n    totals = {}\n    for e in data["expenses"]:\n        totals[e["date"][:7]] = totals.get(e["date"][:7], 0) + e["amount"]\n    if not totals:\n        return []\n    out = []\n    month, last = min(totals), max(totals)\n    while month <= last:\n        out.append((month, totals.get(month, 0)))\n        month = next_month(month)\n    return out\n\n\ndef set_budget(data, category_text, amount_text):\n    """Check both, then set the category\'s monthly budget. Returns (category, pence)."""\n    problems = []\n    category = amount = None\n    try:\n        category = clean_category(category_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        amount = parse_amount(amount_text)\n    except ValueError as e:\n        problems.append(str(e))\n    if problems:\n        raise ValueError("; ".join(problems))\n    data["budgets"][category] = amount\n    return category, amount\n\n\ndef remove_budget(data, category_text):\n    """Remove a category\'s budget and return it. KeyError if it has none."""\n    category = clean_category(category_text)\n    if category not in data["budgets"]:\n        raise KeyError(f"no budget for {category!r}")\n    return data["budgets"].pop(category)\n')
with open('storage.py', 'w', encoding='utf-8') as f:
    f.write('"""Saving and loading the data, and importing and exporting CSV."""\nimport csv\nimport io\nimport json\nimport os\nimport re\nfrom collections import Counter\nfrom datetime import datetime\n\nfrom core import new_data\nfrom money import parse_amount\n\nKEYS = ["id", "date", "amount", "category", "note"]\n\n\ndef load_data(path):\n    """The data saved at path, or new_data() if there\'s no file. ValueError if it\'s damaged."""\n    if not os.path.exists(path):\n        return new_data()\n    with open(path, encoding="utf-8") as f:\n        try:\n            data = json.load(f)\n        except json.JSONDecodeError as e:\n            raise ValueError(f"{path} is damaged: {e.msg} at line {e.lineno}") from None\n    if not (isinstance(data, dict) and isinstance(data.get("expenses"), list)\n            and isinstance(data.get("budgets"), dict) and isinstance(data.get("next_id"), int)):\n        raise ValueError(f"{path} is not a spend data file")\n    for e in data["expenses"]:\n        if not (isinstance(e, dict) and list(e) == KEYS and isinstance(e["amount"], int)):\n            raise ValueError(f"{path} has a damaged expense: {e!r}")\n    return data\n\n\ndef save_data(path, data):\n    """Save safely: write a temporary file, then replace the old file with it in one step."""\n    tmp = path + ".tmp"\n    try:\n        with open(tmp, "w", encoding="utf-8") as f:\n            json.dump(data, f, indent=2, ensure_ascii=False)\n        os.replace(tmp, path)\n    finally:\n        if os.path.exists(tmp):\n            os.remove(tmp)\n\n\nSTATEMENT_AMOUNT = re.compile(r"(-?)(£?[\\d,]+(?:\\.\\d{1,2})?)")\n\n\ndef read_statement(text):\n    """A bank CSV (Date, Description, Amount) as (rows, problems). Only money out is kept."""\n    rows, problems = [], []\n    reader = csv.DictReader(io.StringIO(text))\n    for row in reader:\n        line = reader.line_num\n        try:\n            when = datetime.strptime(row["Date"].strip(), "%d/%m/%Y").date().isoformat()\n        except ValueError:\n            problems.append(f"line {line}: bad date {row[\'Date\'].strip()!r}")\n            continue\n        m = STATEMENT_AMOUNT.fullmatch(row["Amount"].strip())\n        try:\n            if m is None:\n                raise ValueError\n            pence = parse_amount(m.group(2))\n        except ValueError:\n            problems.append(f"line {line}: bad amount {row[\'Amount\'].strip()!r}")\n            continue\n        if m.group(1) != "-":\n            continue\n        rows.append({"date": when, "amount": pence, "description": " ".join(row["Description"].split())})\n    return rows, problems\n\n\nRULES = [\n    (r"tesco|sainsbury|aldi|lidl|co-?op", "food"),\n    (r"\\btfl\\b|trainline|uber|bus", "travel"),\n    (r"netflix|spotify|cinema", "fun"),\n    (r"rent|lettings", "rent"),\n]\n\n\ndef guess_category(description, rules=RULES):\n    for pattern, category in rules:\n        if re.search(pattern, description, re.IGNORECASE):\n            return category\n    return "other"\n\n\ndef import_statement(data, text, rules=RULES):\n    rows, problems = read_statement(text)\n    existing = Counter((e["date"], e["amount"], e["note"]) for e in data["expenses"])\n    added = skipped = 0\n    for r in rows:\n        note = r["description"][:60]\n        key = (r["date"], r["amount"], note)\n        if existing[key] > 0:\n            existing[key] -= 1\n            skipped += 1\n            continue\n        data["expenses"].append({"id": data["next_id"], "date": r["date"], "amount": r["amount"],\n                                 "category": guess_category(r["description"], rules), "note": note})\n        data["next_id"] += 1\n        added += 1\n    return {"added": added, "skipped": skipped, "problems": problems}\n\n\ndef export_csv(data, path):\n    with open(path, "w", newline="", encoding="utf-8") as f:\n        w = csv.writer(f)\n        w.writerow(["id", "date", "amount", "category", "note"])\n        for e in sorted(data["expenses"], key=lambda e: (e["date"], e["id"])):\n            w.writerow([e["id"], e["date"], f"{e[\'amount\'] // 100}.{e[\'amount\'] % 100:02d}", e["category"], e["note"]])\n')
with open('statement.csv', 'w', encoding='utf-8') as f:
    f.write('Date,Description,Amount,Balance\n02/11/2026,TESCO STORES 2041,-62.10,1702.15\n02/11/2026,TFL TRAVEL CH,-4.20,1764.25\n02/11/2026,TFL TRAVEL CH,-4.20,1768.45\n01/11/2026,SALARY,"2,500.00",1772.65\n01/11/2026,CITY LETTINGS RENT,"-3,200.00",-727.35\n31/10/2026,NETFLIX.COM,-10.99,2472.65\n31/10/2026,CORNER CAFE,-3.80,2483.64\n30/10/2026,  PRET   A MANGER ,-6.45,2487.44\n3O/10/2026,UBER *TRIP,-14.00,2493.89\n29/10/2026,AMAZON MKTPLACE,-1O.00,2507.89\n')

### Exercise: Step 26: choose the data file

Write `data_file(arg, environ)`: return `arg` if it isn't `None` or empty; else `environ["SPEND_FILE"]`, stripped, if it's set and not blank; else `spend.json` in the home folder, built with `os.path.join` and `os.path.expanduser("~")`.

In [ ]:
import os

def data_file(arg, environ):
    ___

print(data_file(None, {}))
print(data_file(None, {"SPEND_FILE": "/data/money.json"}))
print(data_file("test.json", {"SPEND_FILE": "/data/money.json"}))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import os as _os\nHOME = _os.path.join(_os.path.expanduser(\"~\"), \"spend.json\")\ndef test_order():\n    \"option, then environment, then default\"\n    assert data_file(\"t.json\", {\"SPEND_FILE\": \"/x.json\"}) == \"t.json\"\n    assert data_file(None, {\"SPEND_FILE\": \"/x.json\"}) == \"/x.json\"\n    assert data_file(None, {\"SPEND_FILE\": \"  /x.json \"}) == \"/x.json\"\n    assert data_file(None, {}) == HOME\ndef test_blanks():\n    \"blank values are ignored\"\n    assert data_file(\"\", {\"SPEND_FILE\": \"/x.json\"}) == \"/x.json\"\n    assert data_file(None, {\"SPEND_FILE\": \"   \"}) == HOME\n    assert data_file(None, {\"OTHER\": \"1\"}) == HOME")

### Exercise: Step 27: find possible duplicates

Write `possible_duplicates(data, days=3)` returning a sorted list of `(lower id, higher id)` pairs of expenses with the same amount whose dates are at most `days` apart. Compare each pair once. `date.fromisoformat` turns `"2026-11-01"` into a `date`.

In [ ]:
from datetime import date, timedelta

def possible_duplicates(data, days=3):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def ex(i, d, a):\n    return {\"id\": i, \"date\": d, \"amount\": a, \"category\": \"x\", \"note\": \"\"}\ndef test_pairs():\n    \"pairs\"\n    data = {\"expenses\": [ex(1, \"2026-11-01\", 320000), ex(5, \"2026-11-01\", 320000), ex(3, \"2026-11-02\", 420),\n                         ex(4, \"2026-11-02\", 420), ex(2, \"2026-10-01\", 320000)], \"budgets\": {}, \"next_id\": 6}\n    assert possible_duplicates(data) == [(1, 5), (3, 4)]\ndef test_window():\n    \"the window, in both directions\"\n    data = {\"expenses\": [ex(7, \"2026-11-04\", 100), ex(2, \"2026-11-01\", 100), ex(9, \"2026-11-08\", 100)], \"budgets\": {}, \"next_id\": 10}\n    assert possible_duplicates(data) == [(2, 7)]\n    assert possible_duplicates(data, days=4) == [(2, 7), (7, 9)]\n    assert possible_duplicates(data, days=0) == []\ndef test_threes_and_none():\n    \"three alike give three pairs; none give none\"\n    data = {\"expenses\": [ex(1, \"2026-11-01\", 50), ex(2, \"2026-11-01\", 50), ex(3, \"2026-11-02\", 50)], \"budgets\": {}, \"next_id\": 4}\n    assert possible_duplicates(data) == [(1, 2), (1, 3), (2, 3)]\n    assert possible_duplicates({\"expenses\": [], \"budgets\": {}, \"next_id\": 1}) == []")

### Exercise: Step 28: change a category

Write `recategorise(data, expense_id, category_text)`: clean the category with `clean_category` (before looking for the expense, so a bad category is reported even for a missing id), change that expense's category, and return `(old, new)`. An unknown id raises `KeyError(f"no expense #{expense_id}")`.

In [ ]:
from money import clean_category

def recategorise(data, expense_id, category_text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def D():\n    return {\"expenses\": [{\"id\": 9, \"date\": \"2026-10-31\", \"amount\": 380, \"category\": \"other\", \"note\": \"CORNER CAFE\"}],\n            \"budgets\": {}, \"next_id\": 10}\ndef test_change():\n    \"changes it\"\n    d = D()\n    assert recategorise(d, 9, \" Eating  Out \") == (\"other\", \"eating out\")\n    assert d[\"expenses\"][0][\"category\"] == \"eating out\"\ndef test_errors():\n    \"errors\"\n    d = D()\n    try:\n        recategorise(d, 4, \"food\")\n        assert False, \"should raise KeyError\"\n    except KeyError as e:\n        assert e.args[0] == \"no expense #4\"\n    try:\n        recategorise(d, 4, \"Food & Drink\")\n        assert False, \"should raise ValueError\"\n    except ValueError as e:\n        assert str(e) == \"bad category: 'Food & Drink'\"\n    assert d == D()")

### Exercise: Step 29: draw the trend

Write `trend_lines(totals, width=30)` for a list of `(month, pence)` pairs (from `monthly_totals`): each line is the month, two spaces, the amount in 11 (right-aligned), two spaces, then the bar, with trailing spaces removed (for a £0 month). No pairs gives `["No expenses."]`. If every month is £0, every bar is empty.

In [ ]:
from money import format_pence

def trend_lines(totals, width=30):
    ___

print("\n".join(trend_lines([("2026-09", 51240), ("2026-10", 332870), ("2026-11", 326210)])))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_basic():\n    \"scaled bars\"\n    assert trend_lines([(\"2026-09\", 51240), (\"2026-10\", 332870), (\"2026-11\", 326210)]) == [\n        \"2026-09      \u00a3512.40  #####\", \"2026-10    \u00a33,328.70  ##############################\",\n        \"2026-11    \u00a33,262.10  #############################\"], trend_lines([(\"2026-09\", 51240), (\"2026-10\", 332870), (\"2026-11\", 326210)])\ndef test_small_and_zero():\n    \"a little spending shows; none shows nothing\"\n    assert trend_lines([(\"2026-10\", 100), (\"2026-11\", 0), (\"2026-12\", 300000)]) == [\n        \"2026-10        \u00a31.00  #\", \"2026-11        \u00a30.00\", \"2026-12    \u00a33,000.00  ##############################\"]\n    assert trend_lines([(\"2026-11\", 0)]) == [\"2026-11        \u00a30.00\"]\ndef test_width_and_empty():\n    \"width, and no data\"\n    assert trend_lines([(\"2026-11\", 500), (\"2026-12\", 1000)], width=10) == [\"2026-11        \u00a35.00  #####\", \"2026-12       \u00a310.00  ##########\"]\n    assert trend_lines([]) == [\"No expenses.\"]")

### Exercise: Step 30: refactor with collect

Write `check_note(text)` (strip it; over 60 characters raises `ValueError("note is longer than 60 characters")`) and `collect(*checks)`, then rewrite `add_expense` and `set_budget` to use them. Behaviour must not change at all: the checks rerun Day 1 and Day 2's tests on your new versions. When you're done, `except ValueError` should appear only once, inside `collect`.

In [ ]:
from money import parse_amount, clean_category
from dates import parse_date


def check_note(text):
    ___


def collect(*checks):
    """Run each (function, argument) check. Return their results, or raise one ValueError listing every problem."""
    ___


def add_expense(data, amount_text, category_text, note, date_text, today):
    """Check every field, then add the expense, or raise one ValueError listing every problem."""
    problems = []
    amount = category = when = None
    try:
        amount = parse_amount(amount_text)
    except ValueError as e:
        problems.append(str(e))
    try:
        category = clean_category(category_text)
    except ValueError as e:
        problems.append(str(e))
    try:
        when = parse_date(date_text, today)
    except ValueError as e:
        problems.append(str(e))
    note = note.strip()
    if len(note) > 60:
        problems.append("note is longer than 60 characters")
    if problems:
        raise ValueError("; ".join(problems))
    expense = {"id": data["next_id"], "date": when, "amount": amount, "category": category, "note": note}
    data["expenses"].append(expense)
    data["next_id"] += 1
    return expense


def set_budget(data, category_text, amount_text):
    """Check both, then set the category's monthly budget. Returns (category, pence)."""
    problems = []
    category = amount = None
    try:
        category = clean_category(category_text)
    except ValueError as e:
        problems.append(str(e))
    try:
        amount = parse_amount(amount_text)
    except ValueError as e:
        problems.append(str(e))
    if problems:
        raise ValueError("; ".join(problems))
    data["budgets"][category] = amount
    return category, amount

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from datetime import date as _date\nT = _date(2026, 11, 2)\ndef _new():\n    return {\"expenses\": [], \"budgets\": {}, \"next_id\": 1}\ndef raises(f, *a):\n    try:\n        f(*a)\n    except ValueError as e:\n        return str(e)\n    raise AssertionError(\"should raise ValueError\")\ndef test_collect():\n    \"collect and check_note\"\n    assert collect((int, \"5\"), (str.upper, \"a\")) == [5, \"A\"]\n    assert raises(collect, (parse_amount, \"x\"), (int, \"3\"), (clean_category, \"1\")) == \"not an amount: 'x'; bad category: '1'\"\n    assert check_note(\"  hi \") == \"hi\" and check_note(\"x\" * 60) == \"x\" * 60\n    assert raises(check_note, \"x\" * 61) == \"note is longer than 60 characters\"\ndef test_add_unchanged():\n    \"add_expense behaves exactly as before\"\n    d = _new()\n    e = add_expense(d, \"12.50\", \" Food \", \"  lunch \", \"\", T)\n    assert e == {\"id\": 1, \"date\": \"2026-11-02\", \"amount\": 1250, \"category\": \"food\", \"note\": \"lunch\"}\n    assert list(e) == [\"id\", \"date\", \"amount\", \"category\", \"note\"] and d[\"next_id\"] == 2\n    assert raises(add_expense, d, \"12.345\", \"Food & Drink\", \"x\" * 61, \"tomorrow\", T) == \\\n        \"not an amount: '12.345'; bad category: 'Food & Drink'; not a date: 'tomorrow'; note is longer than 60 characters\"\n    assert raises(add_expense, d, \"5\", \"food\", \"\", \"2026-12-25\", T) == \"date is in the future: 2026-12-25\"\n    assert d[\"expenses\"] == [e] and d[\"next_id\"] == 2\ndef test_budget_unchanged():\n    \"set_budget behaves exactly as before\"\n    d = _new()\n    assert set_budget(d, \" Eating Out \", \"\u00a345\") == (\"eating out\", 4500) and d[\"budgets\"] == {\"eating out\": 4500}\n    assert raises(set_budget, d, \"Food & Drink\", \"-5\") == \"bad category: 'Food & Drink'; not an amount: '-5'\"\n    assert d[\"budgets\"] == {\"eating out\": 4500}\ndef test_refactored():\n    \"except ValueError appears once\"\n    assert source.count(\"except ValueError\") == 1, \"only collect should catch ValueError\"")

### Exercise: Step 31: add the new commands

The editor has Day 4's `spend.py`, and this exercise has today's functions in `polish.py`. Change it so that:

1. `--file` has no default (help: `the data file (default: $SPEND_FILE, or spend.json in your home folder)`), and `main(argv=None, today=None, environ=None)` loads and saves `data_file(args.file, environ)`, with `os.environ` when `environ` is `None`;
2. `spend category 9 "eating out"` runs `recategorise` and prints `#9: other is now eating out` (and saves);
3. `spend check` prints, for each pair from `possible_duplicates`, a line `Possible duplicate:` and then the two expenses, each as two spaces and its `describe` line; or `No possible duplicates.`;
4. `spend trend` prints `trend_lines(monthly_totals(data))`.

Add the three subcommands after `export` in `build_parser`, each with help, and to `COMMANDS`.

In [ ]:
# Run this cell first: it creates the files used below.
with open('polish.py', 'w', encoding='utf-8') as f:
    f.write('"""Day 5\'s additions."""\nimport os\nfrom datetime import date, timedelta\nfrom money import clean_category, format_pence\n\n\ndef data_file(arg, environ):\n    """The data file: --file if given, else $SPEND_FILE, else spend.json in the home folder."""\n    if arg:\n        return arg\n    if environ.get("SPEND_FILE", "").strip():\n        return environ["SPEND_FILE"].strip()\n    return os.path.join(os.path.expanduser("~"), "spend.json")\n\n\n\ndef possible_duplicates(data, days=3):\n    """Pairs of ids (lower first) with the same amount, dated at most `days` apart, sorted."""\n    pairs = []\n    xs = sorted(data["expenses"], key=lambda e: e["id"])\n    for i, a in enumerate(xs):\n        for b in xs[i + 1:]:\n            if a["amount"] == b["amount"]:\n                gap = abs(date.fromisoformat(a["date"]) - date.fromisoformat(b["date"]))\n                if gap <= timedelta(days=days):\n                    pairs.append((a["id"], b["id"]))\n    return pairs\n\n\n\ndef recategorise(data, expense_id, category_text):\n    """Change one expense\'s category. Returns (old, new). KeyError if there\'s no such expense."""\n    category = clean_category(category_text)\n    for e in data["expenses"]:\n        if e["id"] == expense_id:\n            old = e["category"]\n            e["category"] = category\n            return old, category\n    raise KeyError(f"no expense #{expense_id}")\n\n\n\ndef trend_lines(totals, width=30):\n    """A text bar chart of (month, pence) pairs: the longest bar is `width` characters."""\n    if not totals:\n        return ["No expenses."]\n    top = max(p for _, p in totals)\n    lines = []\n    for month, pence in totals:\n        n = round(pence / top * width) if top else 0\n        if pence and n == 0:\n            n = 1\n        lines.append(f"{month}  {format_pence(pence):>11}  {\'#\' * n}".rstrip())\n    return lines\n')

In [ ]:
"""spend: a command-line expense tracker."""
import argparse
import sys
from datetime import date, datetime

from core import (add_expense, remove_expense, filter_expenses, total, month_summary,
                  set_budget, remove_budget)
from dates import parse_month
from money import format_pence, clean_category
from storage import load_data, save_data, import_statement, export_csv


def describe(e):
    """'#14: £12.50 food (lunch with Tola) on 2026-11-02'"""
    note = f" ({e['note']})" if e["note"] else ""
    return f"#{e['id']}: {format_pence(e['amount'])} {e['category']}{note} on {e['date']}"


def expense_table(expenses):
    """The lines of a table of expenses, with a total line. No expenses gives one line saying so."""
    if not expenses:
        return ["No expenses."]
    lines = [f"{'#':>4}  {'Date':<10}  {'Amount':>11}  {'Category':<12}  Note"]
    for e in expenses:
        lines.append(f"{e['id']:>4}  {e['date']:<10}  {format_pence(e['amount']):>11}  {e['category']:<12}  {e['note']}".rstrip())
    count = "1 expense" if len(expenses) == 1 else f"{len(expenses)} expenses"
    lines.append(f"{'':>4}  {'Total':<10}  {format_pence(total(expenses)):>11}  {count}")
    return lines


def month_name(month):
    """'2026-11' as 'November 2026'."""
    return datetime.strptime(month, "%Y-%m").strftime("%B %Y")


def summary_lines(s):
    """The lines of a month's summary, from month_summary's result."""
    count = "1 expense" if s["count"] == 1 else f"{s['count']} expenses"
    lines = [f"{month_name(s['month'])}: {count}, {format_pence(s['total'])}"]
    for r in s["rows"]:
        line = f"  {r['category']:<12}  {format_pence(r['spent']):>11}"
        if r["budget"] is not None:
            line += f"  of {format_pence(r['budget']):>9}  "
            if r["left"] < 0:
                line += f"OVER by {format_pence(-r['left'])}"
            else:
                line += f"{format_pence(r['left'])} left"
        lines.append(line)
    return lines


def build_parser():
    parser = argparse.ArgumentParser(prog="spend", description="Track what you spend.")
    parser.add_argument("--file", default="spend.json", help="the data file (default: spend.json)")
    sub = parser.add_subparsers(dest="command", required=True)

    p = sub.add_parser("add", help="record an expense")
    p.add_argument("amount", help="for example 12.50 or £1,250")
    p.add_argument("category", help="for example food")
    p.add_argument("note", nargs="?", default="", help="what it was for")
    p.add_argument("--date", default="", help="YYYY-MM-DD, DD/MM/YYYY, today or yesterday")

    p = sub.add_parser("remove", help="remove an expense by its number")
    p.add_argument("id", type=int)

    p = sub.add_parser("list", help="list expenses, newest first")
    p.add_argument("--month", help="YYYY-MM or MM/YYYY")
    p.add_argument("--category")

    p = sub.add_parser("summary", help="a month's spending by category")
    p.add_argument("--month", help="YYYY-MM or MM/YYYY (default: this month)")

    p = sub.add_parser("budget", help="show, set or remove monthly budgets")
    p.add_argument("category", nargs="?")
    p.add_argument("amount", nargs="?", help="an amount, or 'none' to remove the budget")

    p = sub.add_parser("import", help="import a bank statement CSV")
    p.add_argument("statement")

    p = sub.add_parser("export", help="export everything as CSV")
    p.add_argument("out")
    return parser


def cmd_add(data, args, today):
    e = add_expense(data, args.amount, args.category, args.note, args.date, today)
    print("Added", describe(e))
    return True


def cmd_remove(data, args, today):
    print("Removed", describe(remove_expense(data, args.id)))
    return True


def cmd_list(data, args, today):
    month = parse_month(args.month) if args.month else None
    category = clean_category(args.category) if args.category else None
    for line in expense_table(filter_expenses(data, month, category)):
        print(line)
    return False


def cmd_summary(data, args, today):
    month = parse_month(args.month) if args.month else today.isoformat()[:7]
    for line in summary_lines(month_summary(data, month)):
        print(line)
    return False


def cmd_budget(data, args, today):
    if args.category is None:
        if not data["budgets"]:
            print("No budgets.")
        for category, pence in sorted(data["budgets"].items()):
            print(f"  {category:<12}  {format_pence(pence):>11} a month")
        return False
    if args.amount is None:
        raise ValueError("give an amount, or 'none' to remove the budget")
    if args.amount.strip().lower() == "none":
        category = clean_category(args.category)
        remove_budget(data, category)
        print(f"Removed the budget for {category}")
    else:
        category, pence = set_budget(data, args.category, args.amount)
        print(f"Budget for {category}: {format_pence(pence)} a month")
    return True


def cmd_import(data, args, today):
    with open(args.statement, newline="", encoding="utf-8") as f:
        result = import_statement(data, f.read())
    print(f"Imported {result['added']}, skipped {result['skipped']} already there")
    for problem in result["problems"]:
        print(f"  could not read {problem}")
    return result["added"] > 0


def cmd_export(data, args, today):
    export_csv(data, args.out)
    print(f"Exported {len(data['expenses'])} expenses to {args.out}")
    return False


COMMANDS = {"add": cmd_add, "remove": cmd_remove, "list": cmd_list, "summary": cmd_summary,
            "budget": cmd_budget, "import": cmd_import, "export": cmd_export}


def main(argv=None, today=None):
    """Run one command. Returns the exit status: 0 for success, 1 for an error."""
    args = build_parser().parse_args(argv)
    today = today or date.today()
    try:
        data = load_data(args.file)
        if COMMANDS[args.command](data, args, today):
            save_data(args.file, data)
    except (ValueError, KeyError, OSError) as e:
        message = e.args[0] if isinstance(e, KeyError) else str(e)
        print(f"spend: {message}", file=sys.stderr)
        return 1
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import contextlib as _cl, io as _io, json as _json, os as _os\nfrom datetime import date as _date\nT = _date(2026, 11, 2)\ndef go(*argv, environ=None):\n    out, err = _io.StringIO(), _io.StringIO()\n    with _cl.redirect_stdout(out), _cl.redirect_stderr(err):\n        code = main(list(argv), today=T, environ={\"SPEND_FILE\": \"money.json\"} if environ is None else environ)\n    return code, out.getvalue(), err.getvalue()\ndef test_file_choice():\n    \"--file, then SPEND_FILE\"\n    reset_files()\n    go(\"add\", \"5\", \"fun\")\n    assert read_file(\"money.json\") is not None, \"SPEND_FILE wasn't used\"\n    go(\"--file\", \"other.json\", \"add\", \"6\", \"fun\")\n    assert read_file(\"other.json\") is not None and len(_json.loads(read_file(\"money.json\"))[\"expenses\"]) == 1\n    import inspect\n    assert \"environ\" in inspect.signature(main).parameters\ndef test_category():\n    \"category\"\n    reset_files()\n    go(\"import\", \"statement.csv\")\n    assert go(\"category\", \"6\", \"eating out\") == (0, \"#6: other is now eating out\\n\", \"\")\n    assert _json.loads(read_file(\"money.json\"))[\"expenses\"][5][\"category\"] == \"eating out\"\n    assert go(\"category\", \"99\", \"food\") == (1, \"\", \"spend: no expense #99\\n\")\ndef test_check():\n    \"check\"\n    reset_files()\n    assert go(\"check\") == (0, \"No possible duplicates.\\n\", \"\")\n    go(\"add\", \"3,200\", \"rent\", \"--date\", \"2026-11-01\")\n    go(\"import\", \"statement.csv\")\n    code, out, err = go(\"check\")\n    assert out.splitlines() == [\"Possible duplicate:\", \"  #1: \u00a33,200.00 rent on 2026-11-01\",\n                                \"  #5: \u00a33,200.00 rent (CITY LETTINGS RENT) on 2026-11-01\",\n                                \"Possible duplicate:\", \"  #3: \u00a34.20 travel (TFL TRAVEL CH) on 2026-11-02\",\n                                \"  #4: \u00a34.20 travel (TFL TRAVEL CH) on 2026-11-02\"], out\ndef test_trend():\n    \"trend\"\n    reset_files()\n    assert go(\"trend\") == (0, \"No expenses.\\n\", \"\")\n    go(\"import\", \"statement.csv\")\n    assert go(\"trend\")[1] == \"2026-10       \u00a321.24  #\\n2026-11    \u00a33,270.50  ##############################\\n\"\ndef test_help():\n    \"the new commands have help\"\n    out = _io.StringIO()\n    try:\n        with _cl.redirect_stdout(out):\n            build_parser().parse_args([\"--help\"])\n    except SystemExit:\n        pass\n    assert \"{add,remove,list,summary,budget,import,export,category,check,trend}\" in out.getvalue()\n    assert \"SPEND_FILE\" in out.getvalue()")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Using it for real

Using your own program for a day always turns up things the plan missed. For spend, these came up:

1. Typing `--file` every time is tedious, but the data shouldn't land in whatever folder you happen to be in either.
2. Rent was counted twice: once typed, once imported. The import couldn't know they were the same.
3. Imported payments it couldn't place are stuck as `other`, with no way to change them.
4. The summary shows one month; there's no way to see whether spending is going up.
5. `add_expense` and `set_budget` repeat the same `try`/`except` pattern again and again.
6. There's nothing telling a new user how to use it.

Each gets a step today. This is what **polish** means: not decoration, but fixing what real use reveals.

The modules so far are in the files below.

## Step 26: where the data lives

Command-line tools usually find their settings in this order, most specific first:

1. an **option** on the command line (`--file`), for a one-off;
2. an **environment variable** (`SPEND_FILE`), set once in your shell's settings, for every run;
3. a **default** in your home folder, so it's the same file wherever you run it from.

`os.path.expanduser("~")` gives the home folder on any system. The function takes the environment as a parameter rather than reading `os.environ` itself, so tests can pass a plain dictionary instead of changing the real environment.

## Step 27: possible duplicates

The double rent can't be caught at import time without risking the opposite mistake: yesterday's two bus fares had the same date, amount and description, and both were real. No rule can tell those cases apart; a person can. So spend shouldn't guess. It should **point out** likely duplicates and let you remove the ones that really are.

"Likely" here: the same amount, dated at most 3 days apart. To find them, compare **every pair** of expenses once, with two loops where the inner one starts after the outer, as in selection sort (Week 18). For `n` expenses that's about n²/2 comparisons: fine for the few thousand expenses a person has (Week 17's lesson that O(n²) can be perfectly good for small n).

## Step 28: fixing a category

## Step 29: a trend chart

A chart in a terminal can be made of characters. Each month gets a bar of `#`s whose length is its total **scaled** so the biggest month fills the width: length = total ÷ largest total × width, rounded. One detail matters for honesty: a month with a little spending rounds to 0 characters and looks like nothing at all, so any month above £0 gets at least one `#`.

```
2026-09      £512.40  #####
2026-10    £3,328.70  ##############################
2026-11    £3,262.10  #############################
```

## Step 30: refactoring the checks

`add_expense` and `set_budget` repeat the same pattern: run a check inside `try`, collect its message on failure, then raise them all at once. Repetition like this tends to drift: one day someone fixes the pattern in one place and forgets the other. Day 2's tests mean you can now tidy it **safely**: change the structure, run the tests, and know straight away if behaviour changed (Week 24, Day 4).

The new helper, `collect`, takes `(check, argument)` pairs, runs every check, and returns all their results, or raises one `ValueError` listing every problem. The note check becomes a function too, so all four fields work the same way. `parse_date` needs `today` as well as the text, so it's wrapped in a `lambda` that takes only the text.

Fifty lines became twenty, and adding a fifth field to an expense now means adding one line. The tests passed before and after, which is the only evidence that matters: the structure changed and the behaviour didn't.

## Step 31: new commands

Now wire the new pieces into `spend.py`. This is what most programming work looks like: not a blank file, but an existing program you change carefully, in a few places, without breaking what's there.

## A README

A program without instructions is only usable by the person who wrote it, and only while they remember. Every project gets a **README**: a file called `README.md` at the top of the folder, written in Markdown, which GitHub shows on the project's front page (Week 21). Keep it short and practical: what it is, how to install and run it, how to use it, and how to run the tests.

````
# spend

A small command-line expense tracker. Record what you spend, set monthly
budgets, import bank statements, and see where the money goes.

## Running it

Needs Python 3.10 or newer, and nothing else. In the project folder:

```
python3 spend.py add 12.50 food "lunch with Tola"
python3 spend.py summary
```

Your data is saved in `spend.json` in your home folder. To use another
file, set `SPEND_FILE` or pass `--file`.

## Commands

| Command | What it does |
|---|---|
| `add AMOUNT CATEGORY [NOTE] [--date DATE]` | record an expense |
| `remove ID` | remove an expense |
| `list [--month M] [--category C]` | list expenses, newest first |
| `summary [--month M]` | a month by category, with budgets |
| `budget [CATEGORY [AMOUNT or none]]` | show, set or remove budgets |
| `import FILE` | import a bank statement CSV |
| `export FILE` | export everything as CSV |
| `category ID CATEGORY` | change an expense's category |
| `check` | list possible duplicates |
| `trend` | spending month by month |

`python3 spend.py --help` and `python3 spend.py add --help` give details.

## Tests

```
pip install pytest
pytest
```
````

Notice what it doesn't include: how the code works inside. That's what the docstrings and the code are for. The README is for someone who wants to **use** it.

## Quiz

## Before you go

Add today's changes to your `spend` folder (put the Step 26 to 29 functions where you think they belong: `core.py`, a new module, or `spend.py`), write the README, run `pytest`, and commit. Tomorrow you'll review the whole project: what went well, what you'd change, and where it could go next.